In [6]:
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime

import geopandas as gpd
from shapely.geometry import Point

# --------------------------------------------------
# Paths
# --------------------------------------------------

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --------------------------------------------------
# Helpers
# --------------------------------------------------

def data_quality_report(df, name):
    return {
        "dataset": name,
        "rows": len(df),
        "missing_values": df.isna().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "columns": list(df.columns)
    }


def clean_dataframe(df):
    df = df.drop_duplicates()

    required = ["latitude", "longitude"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    df = df.dropna(subset=required)

    df = df[
        df["latitude"].between(-90, 90) &
        df["longitude"].between(-180, 180)
    ]

    return df


def add_geometry(df):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=[Point(xy) for xy in zip(df["longitude"], df["latitude"])],
        crs="EPSG:4326"
    )
    return gdf


def engineer_features(gdf):
    df = gdf.copy()

    # -----------------------------
    # spatial numeric features
    # -----------------------------
    df["lat"] = df.geometry.y
    df["lon"] = df.geometry.x

    # -----------------------------
    # brightness / frp
    # -----------------------------
    for col in ["brightness", "frp"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # -----------------------------
    # time encoding (IMPORTANT for DBSCAN)
    # instead of strings → cyclic encoding
    # -----------------------------
    if "acq_time" in df.columns:
        t = (
            df["acq_time"]
            .astype(str)
            .str.zfill(4)
        )

        hours = pd.to_numeric(t.str[:2], errors="coerce")
        minutes = pd.to_numeric(t.str[2:], errors="coerce")

        total_minutes = hours * 60 + minutes

        df["time_sin"] = np.sin(2 * np.pi * total_minutes / (24 * 60))
        df["time_cos"] = np.cos(2 * np.pi * total_minutes / (24 * 60))

    # -----------------------------
    # date encoding (optional)
    # -----------------------------
    if "acq_date" in df.columns:
        dt = pd.to_datetime(df["acq_date"], errors="coerce")
        df["day_of_year"] = dt.dt.dayofyear
        df["date_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
        df["date_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

    return df


def build_dbscan_features(df):
    feature_cols = [
        "lat",
        "lon",
        "brightness",
        "frp",
        "time_sin",
        "time_cos",
        "date_sin",
        "date_cos"
    ]

    return df[[c for c in feature_cols if c in df.columns]].copy()


# --------------------------------------------------
# Pipeline
# --------------------------------------------------

reports = []

csv_files = glob.glob(os.path.join(RAW_DIR, "*.csv"))

if not csv_files:
    print("Keine CSV Dateien gefunden.")

for file_path in csv_files:

    name = os.path.basename(file_path)

    # -----------------------------
    # Load
    # -----------------------------
    df = pd.read_csv(file_path)

    # -----------------------------
    # Clean
    # -----------------------------
    df = clean_dataframe(df)

    # -----------------------------
    # Geo
    # -----------------------------
    gdf = add_geometry(df)

    # -----------------------------
    # Feature Engineering
    # -----------------------------
    features_df = engineer_features(gdf)

    # -----------------------------
    # DBSCAN Features
    # -----------------------------
    dbscan_features = build_dbscan_features(features_df)

    # -----------------------------
    # Quality report
    # -----------------------------
    report = data_quality_report(df, name)

    for col in ["brightness", "frp"]:
        if col in features_df.columns:
            report[f"{col}_min"] = features_df[col].min()
            report[f"{col}_max"] = features_df[col].max()

    reports.append(report)

    # -----------------------------
    # Save outputs
    # -----------------------------
    base = name.replace(".csv", "")

    gdf.to_file(
        os.path.join(PROCESSED_DIR, f"{base}_geo.geojson"),
        driver="GeoJSON"
    )

    dbscan_features.to_csv(
        os.path.join(PROCESSED_DIR, f"{base}_dbscan.csv"),
        index=False
    )

# --------------------------------------------------
# Save report
# --------------------------------------------------

report_df = pd.DataFrame(reports)

report_df.to_csv(
    os.path.join(
        PROCESSED_DIR,
        f"data_quality_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    ),
    index=False
)

print("Pipeline fertig.")

Pipeline fertig.
